In [ ]:
import os
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
def deed(stock):
    global concatList
    return np.load(f'./DataAfterPreProcessing/{stock}/x_train.npy')
concatList = np.vstack(Parallel(n_jobs=-1)(delayed(deed)(stock) for stock in tqdm(os.listdir("./DataAfterPreProcessing")[6000:12000])))

In [ ]:
import os
len(os.listdir("./DataAfterPreProcessing"))

In [ ]:
import numpy as np
import os
from joblib import Parallel, delayed
from tqdm import tqdm

In [ ]:
concatList.shape

In [ ]:
np.save('./numpyFinalSaves/x_train2.npy', concatList)

In [ ]:
import numpy as np
x_train = np.vstack([np.load('./numpyFinalSaves/x_train1.npy'), np.load('./numpyFinalSaves/x_train2.npy'), np.load('./numpyFinalSaves/x_train3.npy')])

In [ ]:
x_train.shape

In [ ]:
test = np.array(np.load(f'./DataAfterPreProcessing/{"AAPL"}/x_train.npy'))

In [ ]:
len(results)

In [ ]:
test = np.vstack([test,np.load(f'./DataAfterPreProcessing/{"AAPL"}/x_train.npy')])

In [ ]:
x_train12 = np.vstack([np.load(f'./numpyFinalSaves/x_train1.npy'), np.load(f'./numpyFinalSaves/x_train2.npy')])

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
def auditLoop(i, row):
    if len(np.unique(row.T[-1])) < 15:
        return i
    return np.nan
one = np.load('./numpyFinalSaves/x_train1.npy')
results = np.array(Parallel(n_jobs=-1)(delayed(auditLoop)(i, row) for i, row in tqdm(enumerate(one))))
results = results[~np.isnan(results)]
one = np.delete(one, results.astype('int'))
np.save('./numpyFinalSaves/x_train1Audit.npy', one)
del one
two = np.load('./numpyFinalSaves/y_train1.npy')
two = np.delete(two, results.astype('int'))
np.save('./numpyFinalSaves/y_train2Audit.npy', two)

In [ ]:
one[129].T[0]

In [ ]:
results.dtype

In [ ]:
len(results)

In [ ]:
list(set(one[0].T[0]) - set(np.unique(one[0].T[0])))


In [ ]:
np.unique(one[0].T[0])

In [ ]:
one[1].T[0]

In [ ]:
import pandas as pd 
from tqdm import tqdm
data = pd.read_feather('./data/BigTestDataset/CLOSEDATAFINAL.feather')['Close']
dataHigh = pd.read_feather('./data/BigTestDataset/HIGHDATAFINAL.feather')['High']
dataLow = pd.read_feather('./data/BigTestDataset/LOWDATAFINAL.feather')['Low']
def calculate_stochastic_oscillator(close,low, high, lookback_period=14, d_period=3):
    lowestLow = low.rolling(window=lookback_period).min()
    highestHigh = high.rolling(window=lookback_period).max()
    K = ((close - lowestLow) / (highestHigh -lowestLow)) * 100
    D = K.rolling(window=d_period).mean()
    return D
for column in tqdm(data.columns):
    data[column] = calculate_stochastic_oscillator(data[column], dataHigh[column], dataLow[column])

In [ ]:
import pandas as pd
import os
concatDataframes = []
options = os.listdir('./data/BigTestDataset')
for option in options:
    close = pd.read_feather(f'./data/BigTestDataset/{option}', columns=['Date', "AAPL"])
    close = close.rename(columns={'AAPL': option.split(".feather")[0]})
    concatDataframes.append(close)
dataframe = pd.concat(concatDataframes, axis=1).dropna().T.values
del concatDataframes, close

In [ ]:
dataframe

In [ ]:
len(dataframe)

In [ ]:
dataframe[0]

In [ ]:
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import numpy as np
def preprocessFunction(column):
    concatDataframes = []
    options = os.listdir('./data/BigTestDataset')
    for option in options:
        close = pd.read_feather(f'./data/BigTestDataset/{option}', columns=['Date', f"{column}"])
        close = close.rename(columns={f'{column}': option.split(".feather")[0]})
        concatDataframes.append(close)
    dataframe = pd.concat(concatDataframes, axis=1).dropna().T.values
    if(len(dataframe[0])<180):
        return
    del concatDataframes, close
    data = []
    y = []
    scalar = MinMaxScaler(feature_range=(-1,1))
    for index in range(len(dataframe[0])-21):
        subData = []
        subData.append(scalar.fit_transform(dataframe[-1][index:index+20].reshape(-1,1)))
        y.append(scalar.transform(dataframe[-1][index+21].reshape(-1,1)))
        for listOfData in dataframe[[0,1,3,4,5,7,8,9,12]]:
            subData.append(scalar.transform(listOfData[index:index+20].reshape(-1,1)))
            # raise Exception(scalar.transform(listOfData[index:index+21].reshape(-1,1)).shape)
        subData.append(scalar.fit_transform(dataframe[2][index:index+20].reshape(-1,1)))
        subData.append(scalar.fit_transform(dataframe[6][index:index+20].reshape(-1,1)))
        subData.append(scalar.fit_transform(dataframe[10][index:index+20].reshape(-1,1)))
        subData.append(scalar.transform(dataframe[11][index:index+20].reshape(-1,1)))
        data.append(np.transpose(np.array(subData), (1,0,2)))
    data = np.array(data).squeeze()
    y = np.array(y).squeeze(axis=1)
    x_train, x_test, y_train, y_test = train_test_split(data, y, test_size=.2, shuffle=False)
    Path(f'./DataAfterPreProcessing/{column}').mkdir(exist_ok=True)
    np.save(f'./DataAfterPreProcessing/{column}/x_train.npy', x_train)
    np.save(f'./DataAfterPreProcessing/{column}/x_test.npy', x_test)
    np.save(f'./DataAfterPreProcessing/{column}/y_train.npy', y_train)
    np.save(f'./DataAfterPreProcessing/{column}/y_test.npy', y_test)
from joblib import Parallel, delayed
columns = pd.read_feather('./data/BigTestDataset/CLOSEDATAFINAL.feather').columns
Parallel(n_jobs=-1)(
    delayed(preprocessFunction)(column) for column in tqdm(columns)
)

In [ ]:
from joblib import Parallel, delayed
columns = pd.read_feather('./data/BigTestDataset/CLOSEDATAFINAL.feather').columns
Parallel(n_jobs=-1)(
    delayed(preprocessFunction)(column) for column in tqdm(columns)
)

In [ ]:
x_train.shape

In [ ]:
np.array

In [ ]:
np.array(y).squeeze(axis=1)

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import torch

def add_feature_engineering(df, stock):
    price = df.reshape(-1,1)
    scaler = MinMaxScaler(feature_range=(-1,1))
    def split_data(stock, lookback):
        data = []
        for index in range(len(stock) - lookback):
            data.append(scaler.fit_transform(stock[index: index + lookback]))
        data = np.array(data)
        test_set_size = int(np.round(0.2*data.shape[0]))
        train_set_size = data.shape[0] - (test_set_size)
        x_train = data[:train_set_size,:-1,:]
        y_train = data[:train_set_size,-1,:]
        x_test = data[train_set_size:,:-1]
        y_test = data[train_set_size:,-1,:]
        return [x_train, y_train, x_test, y_test]
    lookback = 20 # choose sequence length
    x_train, y_train, x_test, y_test = split_data(price, lookback)
    return x_train, y_train, x_test, y_test
data = pd.read_feather('./FULLDATASETFINALOPEN.feather')
results = []
for stock in ['AAPL']:
# for stock in tqdm(data.columns[20:]):
    values = data[stock].to_numpy()
    values = values[~np.isnan(values)]
    if(len(values) < 180):
        continue
    x_train, y_train, x_test, y_test = add_feature_engineering(values, stock)
    for i, value in enumerate(x_train):
        if(len(np.unique([zero[0] for zero in value])) < 15):
            y_train[i] = np.nan
            x_train[i] = np.nan
    if len(x_train) != 0:
        results.append((x_train, y_train, x_test, y_test))


In [ ]:
x_train = np.concatenate([result[0] for result in results])
x_test = np.concatenate([result[2] for result in results])
y_train = np.concatenate([result[1] for result in results])
y_test = np.concatenate([result[3] for result in results])
from torch.utils.data import TensorDataset, DataLoader
x_train = torch.from_numpy(x_train).type(torch.Tensor)
x_test = torch.from_numpy(x_test).type(torch.Tensor)
y_train_gru = torch.from_numpy(y_train).type(torch.Tensor)
y_test_gru = torch.from_numpy(y_test).type(torch.Tensor)
dataset = TensorDataset(x_train, y_train_gru)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
dataset_test = TensorDataset(x_test, y_test_gru)
dataloader_test = DataLoader(dataset_test, batch_size=32, shuffle=False)

torch.save(dataloader, './dataloaderNormlaized.pt')
torch.save(dataloader_test, './dataloader_testNormalzied.pt')

In [ ]:
import torch
dataloader = torch.load('./dataloaderNormlaized.pt')

In [ ]:
(next(iter(dataloader)))

In [ ]:
import polars as pl
import pandas as pd
data = pl.from_pandas(pd.read_feather('FULLTESTER.feather')['Close'])
for row in data.iter_rows():
    print(row)
    break

In [ ]:
import yfinance as yf
import os
from datetime import datetime, timedelta
stocks = [stock2.split('.pt')[0] for stock2 in os.listdir('Models')]
data = yf.Tickers(stocks).download(start=(datetime(2023,1,1)-timedelta(365)).strftime('%Y-%m-%d'), end=datetime(2023,1,1).strftime('%Y-%m-%d'), repair=True, progress=True)
pd.read_

In [ ]:
AllColumns = []
for column in data.columns:
    if column[0] not in AllColumns:
        AllColumns.append(column[0])

In [ ]:
AllColumns

In [ ]:
import yfinance as yf
from datetime import datetime, timedelta
import os
stocks = [stock2.split('.pt')[0] for stock2 in os.listdir('Models')]
stocks.remove('HUBB')
currentTime= datetime(2023,1,4)

data = yf.Tickers(stocks[:10]).download(start=(currentTime-timedelta(10)).strftime('%Y-%m-%d'), end=currentTime.strftime('%Y-%m-%d'), repair=True)
data['Close'].iloc[-1]

In [ ]:
import pandas as pd
import os
stocks = [stock2.split('.feather')[0] for stock2 in os.listdir('AllData')]
stockPriceValues = pd.concat((pd.read_feather(f'./AllData/{stock}.feather').rename(columns={'Close':stock})[stock] for stock in stocks), axis=1)
stockPriceValues.columns = pd.MultiIndex.from_tuples([('Close', stock) for stock in stocks])

In [ ]:
stockPriceValues.to_feather('./FULLTESTER.feather')

In [ ]:
import numpy as np
data = stockPriceValues.loc[(pd.DatetimeIndex(stockPriceValues.index).tz_convert(None) > np.datetime64(pd.DatetimeIndex([datetime(2023,1,1)-timedelta(days=365)])[0])) & (pd.DatetimeIndex(stockPriceValues.index).tz_convert(None) < np.datetime64(pd.DatetimeIndex([datetime(2023,1,1)])[0]))].dropna(axis=1)

In [ ]:
import pandas as pd
pd.read_feather('FULLTESTER.feather').sort_index()

In [ ]:
import os
from alpaca_trade_api import REST
API_KEY = os.environ.get("API_KEY", "")
API_SECRET = os.environ.get("API_SECRET", "")
BASE_URL = "https://paper-api.alpaca.markets"
ALPACA_CREDS = {
    "API_KEY": API_KEY,
    "API_SECRET": API_SECRET,
    "PAPER": True,
}
api = REST(base_url=BASE_URL, key_id=API_KEY, secret_key=API_SECRET)
stocks = api.list_assets()

In [ ]:
stocks = [stock.__dict__['_raw']['symbol'] for stock in stocks if stock.__dict__['_raw']['class'] == 'us_equity']

In [ ]:
import yfinance as yf
data = yf.Tickers(stocks[2*(len(stocks)//4):3*(len(stocks)//4)]).download(period="max", repair=True, progress=True)[['Open', "High", "Low", "Close", "Volume"]]

In [ ]:
data

In [ ]:
from pyarrow import feather
feather.write_feather(data, "FULLDATASET3.feather")

In [ ]:
import pandas as pd
one = pd.read_feather('FULLDATASET1-2.feather')
two = pd.read_feather('FULLDATASET3-4.feather')
four = pd.concat([one, two], axis=0)
del one, two
from pyarrow import feather
feather.write_feather(four, "FULLDATASETFINALCLOSEOPEN.feather")

In [ ]:
import numpy as np
import pandas as pd
three = pd.read_feather('./data/FULLDATASETFINALCLOSEOPEN.feather')
three = three.sort_index().groupby(level=0).sum().replace(0, np.nan)

In [ ]:
from pyarrow import feather
feather.write_feather(three, "./data/FULLDATASETFINALCLOSEOPEN.feather")

In [ ]:
data = feather.read_feather('./FULLDATASETFINALOPEN.feather')
temp3 = [x for x in stocks if x not in list(data.columns)]


In [ ]:
len(stocks)- len(data.columns), len(temp3)

In [ ]:
import pandas as pd
from datetime import datetime
import yfinance as yf
import numpy as np
dateUsed = datetime(2022,1,4)
data = yf.Tickers(stocks).download(start=(dateUsed-timedelta(365)).strftime('%Y-%m-%d'), end=dateUsed.strftime('%Y-%m-%d'), repair=True, progress=False)
stocks = [stock2.split('.pt')[0] for stock2 in os.listdir('Models')]
stocks.remove('HUBB')
stockPriceValues = pd.concat((pd.read_feather(f'./AllData/{stock}.feather').rename(columns={'Close':stock})[stock] for stock in stocks), axis=1)
stockPriceValues.columns = pd.MultiIndex.from_tuples([('Close', stock) for stock in stocks])
data2 = stockPriceValues.loc[(stockPriceValues.index.tz_convert(None) > np.datetime64(pd.DatetimeIndex([dateUsed-timedelta(days=365)])[0])) & (stockPriceValues.index.tz_convert(None) < np.datetime64(pd.DatetimeIndex([dateUsed])[0]))]

In [ ]:
data = yf.Tickers(np.setdiff1d(list(data2.dropna(axis=1)['Close'].columns), list(data['Close'].dropna(axis=1).columns)).tolist()).download(start=(dateUsed-timedelta(365)).strftime('%Y-%m-%d'), end=dateUsed.strftime('%Y-%m-%d'), repair=True, progress=False)

In [ ]:
import pandas as pd
import glob
import os
stocks = [stock2.split('.pt')[0] for stock2 in os.listdir('Models')]
stocks.remove('HUBB')

df = pd.concat((pd.read_feather(f'./AllData/{stock}.feather').rename(columns={'Close':stock})[stock] for stock in stocks), axis=1)

In [ ]:
data['Close'].dropna(axis=1).columns

In [ ]:
import pandas as pd
pd.read_feather('./AllData/AAPL.feather').rename(columns={'Close':'AAPL'})['AAPL']

In [ ]:
df.columns = pd.MultiIndex.from_tuples([('Close', stock) for stock in stocks])

In [ ]:
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import os
stocks = [stock2.split('.pt')[0] for stock2 in os.listdir('Models')]
stocks.remove('HUBB')

df = pd.concat((pd.read_feather(f'./AllData/{stock}.feather').rename(columns={'Close':stock})[stock] for stock in stocks), axis=1)
pd.read_feather('./AllData/AAPL.feather').rename(columns={'Close':'AAPL'})['AAPL']
df.columns = pd.MultiIndex.from_tuples([('Close', stock) for stock in stocks])
df.loc[(df.index.tz_convert(None) > np.datetime64(pd.DatetimeIndex([datetime(2023,1,1)-timedelta(days=365)])[0])) & (df.index.tz_convert(None) < np.datetime64(pd.DatetimeIndex([datetime(2023,1,1)])[0]))]

In [ ]:
import pandas as pd
import numpy as np
from pyarrow.feather import write_feather
import os
import torch
import ray
from sklearn.preprocessing import MinMaxScaler
import yfinance as yf
import datetime
def getStockPrices(currentTime, stock):
        try:
            data = yf.Ticker(stock).history(period='1mo')
            data.index = pd.to_datetime(data.index).strftime('%Y-%B-%d')
            target_loc = (data.index.get_loc(currentTime.strftime('%Y-%B-%d')))
            start_loc = max(0, target_loc - 19)
            selected_rows = data.iloc[start_loc:target_loc + 1]
            reshapedRows = (selected_rows['Close'].values.reshape(-1,1))
            scaler = MinMaxScaler(feature_range=(-1,1))
            data = torch.from_numpy((np.array(([scaler.fit_transform(reshapedRows)])))).type(torch.Tensor)
            return (data, scaler)
        except:
            return (None, None)
getStockPrices(datetime.datetime.now(), 'AAPL')
# volatility('AAPL')
# stocks = [stock2.split('.feather')[0] for stock2 in os.listdir('AllData')]
# ray.get([volatility.remote(stock) for stock in stocks])

In [ ]:
df = pd.read_feather('./VolatilityAllData/GOOGL.feather')
df

In [ ]:
import pandas as pd
import numpy as np
from functools import reduce
import os
import torch
from sklearn.preprocessing import MinMaxScaler
import joblib
import polars as pl
from datetime import datetime
start_date = datetime(2020,1,1)
import torch
import torch.nn as nn
device= "cpu"
print(device)
def add_feature_engineering(df, stock):
    price = df.reshape(-1,1)
    scaler = MinMaxScaler(feature_range=(-1,1))
    def split_data(stock, lookback):
        data = []
        for index in range(len(stock) - lookback):
            data.append(scaler.fit_transform(stock[index: index + lookback]))
        data = np.array(data)
        test_set_size = int(np.round(0.2*data.shape[0]))
        train_set_size = data.shape[0] - (test_set_size)
        x_train = data[:train_set_size,:-1,:]
        y_train = data[:train_set_size,-1,:]
        x_test = data[train_set_size:,:-1]
        y_test = data[train_set_size:,-1,:]
        return [x_train, y_train, x_test, y_test]
    lookback = 20 # choose sequence length
    x_train, y_train, x_test, y_test = split_data(price, lookback)
    return x_train, y_train, x_test, y_test


def create_full_dataset(stock):
  data = pd.read_feather(f'./AllData/{stock}.feather')
  data = data.loc[(data.index.tz_localize(None) < np.datetime64(start_date))]
  x_train, y_train, x_test, y_test = add_feature_engineering(data, stock)
  x_train = torch.from_numpy(x_train).type(torch.Tensor).to(device)
  x_test = torch.from_numpy(x_test).type(torch.Tensor).to(device)
  y_train_gru = torch.from_numpy(y_train).type(torch.Tensor).to(device)
  y_test_gru = torch.from_numpy(y_test).type(torch.Tensor)

  return x_train, x_test, y_train_gru, y_test_gru
# get all values 
# stocks = [stock for stock in [stock1.split('.feather')[0] for stock1 in os.listdir('AllData')] if stock not in [stock2.split('.pt')[0] for stock2 in os.listdir('Models')]]
# stocks = [stock2.split('.pt')[0] for stock2 in os.listdir('Models')]
# stocks.remove('HUBB')
# futures = [create_full_dataset.remote(stock) for stock in stocks]
# results = ray.get(futures)
# hist, model, scaler = create_full_dataset('AAPL')
# data = pd.read_feather(f'./AllData/{"AAPL"}.feather')
# data = data.loc[(data.index.tz_localize(None) < np.datetime64(start_date))]
# x_train, y_train, x_test, y_test, scaler = add_feature_engineering(data)
# x_train = torch.from_numpy(x_train).type(torch.Tensor).to(device)
# print(x_train)
# print(x_train[:,:,0][0])
print("_________________FINISHED___________________")
# stock = "AAPL"
# df = pl.read_ipc(f'./CombinedDataframe/{stock}.feather')
# x_train, y_train, x_test, y_test, scaler = add_feature_engineering(df)
"""
input_dim = 1
  hidden_dim = 32
  num_layers = 2
  output_dim = 1
  num_epochs = 100
  class GRU(nn.Module):
      def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
          super(GRU, self).__init__()
          self.hidden_dim = hidden_dim
          self.num_layers = num_layers

          self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True)
          self.fc = nn.Linear(hidden_dim, output_dim)

      def forward(self, x):
          h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).requires_grad_()
          out, (hn) = self.gru(x, (h0.detach()))
          out = self.fc(out[:, -1, :])
          return out
  model = GRU(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers).to(device)
  criterion = torch.nn.MSELoss(reduction='mean')
  optimiser = torch.optim.Adam(model.parameters(), lr=0.01)
  hist = np.zeros(num_epochs)
  for t in range(num_epochs):
      y_train_pred = model(x_train)
      loss = criterion(y_train_pred, y_train_gru)
      hist[t] = loss.item()
      optimiser.zero_grad()
      loss.backward()
      optimiser.step()
  torch.save(model.state_dict(), f'./{stock}.pt')
  """

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
data = pd.read_feather('./FULLDATASETFINALOPEN.feather')
results = []
coutner = 0
for stock in ['AAPL']:
# for stock in tqdm(data.columns):
    values = data[stock].to_numpy()
    values = values[~np.isnan(values)]
    if(len(values) < 180):
        coutner += 1
        continue
    x_train, y_train, x_test, y_test = add_feature_engineering(values, stock)
    results.append((x_train, y_train, x_test, y_test))

In [ ]:
results[0][0][0]

In [ ]:
import torch
dataloader = torch.load('./dataloader.pt')
dataloader = (dataloader)

In [ ]:
(next(dataloader))

In [ ]:
x_train = np.concatenate([result[0] for result in results])
x_test = np.concatenate([result[2] for result in results])
y_train = np.concatenate([result[1] for result in results])
y_test = np.concatenate([result[3] for result in results])

In [ ]:
for data in x_train:
    if data.std() == 0:
        print('HELLO')

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
x_train = torch.from_numpy(x_train).type(torch.Tensor)
x_test = torch.from_numpy(x_test).type(torch.Tensor)
y_train_gru = torch.from_numpy(y_train).type(torch.Tensor)
y_test_gru = torch.from_numpy(y_test).type(torch.Tensor)
dataset = TensorDataset(x_train, y_train_gru)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
dataset_test = TensorDataset(x_test, y_test_gru)
dataloader_test = DataLoader(dataset_test, batch_size=32, shuffle=False)

In [ ]:
x_train[2000]

In [ ]:
torch.save(dataloader, './dataloader.pt')
torch.save(dataloader_test, './dataloader_test.pt')

In [ ]:
from torch import nn
import numpy as np
input_dim = 1
hidden_dim = 32
num_layers = 2
output_dim = 1
import torch
dataloader = torch.load('./dataloader.pt')
dataloader_test = torch.load('./dataloaderTest.pt')
num_epochs = 100
from tqdm import tqdm
class GRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(GRU, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=input_dim, out_channels=16, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        self.gru = nn.GRU(16, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, output_dim)
        )
    def forward(self, x):
        x = x.permute(0, 2, 1) 
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).requires_grad_().to(device='cuda')
        out, (hn) = self.gru(x, (h0.detach()))
        out = self.fc(out[:, -1, :])
        return out
model = GRU(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers).to(device='cuda')
criterion = torch.nn.MSELoss(reduction='mean')
optimiser = torch.optim.Adam(model.parameters(), lr=1e-6, weight_decay=1e-5)
hist = np.zeros(num_epochs*32)
for t in range(num_epochs):
    total_train_loss = 0
    for data in tqdm(dataloader):
        x_train, y_train = data
        y_train_pred = model(x_train.to('cuda'))
        loss = criterion(y_train_pred, y_train.to('cuda'))
        total_train_loss += loss
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        del x_train, y_train, loss
    print(total_train_loss)
    total_train_loss /= len(dataloader)
    total_loss = 0
    with torch.inference_mode():
        for data in tqdm(dataloader_test):
            x_train, y_train = data
            y_train_pred = model(x_train.to('cuda'))
            loss = criterion(y_train_pred, y_train.to('cuda'))
            total_loss += loss
            del x_train, y_train, loss
        total_loss /= len(dataloader_test)
    hist[t] = total_loss
    print(f"Epoch: {t} | Test Loss: {total_loss} | Train Loss: {total_train_loss} | Difference: {total_loss - total_train_loss}")
    torch.save(model.state_dict(), f'./{"FULL2"}.pt')
    if(abs(total_loss - total_train_loss) < 3e-7):
        break


In [ ]:
torch.save(model.state_dict(), f'./{"FULL3"}.pt')

In [ ]:
len(hist)

In [ ]:
total_loss

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
sns.set_style("darkgrid")

fig = plt.figure()
fig.subplots_adjust(hspace=0.2, wspace=0.2)

# plt.subplot(1, 2, 1)
# ax = sns.lineplot(x = original.index, y = original[0], label="Data", color='royalblue')
# ax = sns.lineplot(x = predict.index, y = predict[0], label="Training Prediction (GRU)", color='tomato')
# ax.set_title('Stock price', size = 14, fontweight='bold')
# ax.set_xlabel("Days", size = 14)
# ax.set_ylabel("Cost (USD)", size = 14)
# ax.set_xticklabels('', size=10)


plt.subplot(1, 2, 1)
ax = sns.lineplot(data=hist, color='royalblue')
ax.set_xlabel("Epoch", size = 14)
ax.set_ylabel("Loss", size = 14)
ax.set_title("Training Loss", size = 14, fontweight='bold')
fig.set_figheight(6)
fig.set_figwidth(16)

In [ ]:
import joblib


In [ ]:
import yfinance as yf
import ray
from pyarrow import feather
api = REST(base_url=BASE_URL, key_id=API_KEY, secret_key=API_SECRET)
assets = api.list_assets(status="active")
assetsNotCrypto = [asset for asset in assets if asset._raw['class'] != 'crypto']
tickers = [asset._raw['symbol'] for asset in assetsNotCrypto]
tickers = list(set(tickers))
@ray.remote
def getData(stock):
  history = yf.Ticker(stock).history(period='max')
  feather.write_feather(history, f'./AllData/{stock}.feather')
futures = [getData.remote(ticker) for ticker in tickers]
results = ray.get(futures)

In [ ]:
torch.save(model.state_dict(), f'Models/{stock}.pt')
joblib.dump(scaler, f'Scalars/{stock}.gz')

# Back Tester

In [ ]:
%pip install ray

In [ ]:
def initialModels(stock):
    data = pd.read_feather(f'./AllData/{stock}.feather')
    data = data.loc[(data.index.tz_localize(None) < np.datetime64(start_date))]
    
initialModels('AAPL')

In [ ]:
from datetime import datetime
from lumibot.strategies import Strategy
from lumibot.traders import Trader
from lumibot.brokers import Alpaca
from lumibot.backtesting import YahooDataBacktesting
from torch import nn
from alpaca_trade_api import REST
import pandas as pd
import os
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
import joblib

In [ ]:
input_dim = 1
hidden_dim = 32
num_layers = 2
output_dim = 1
num_epochs = 100
class GRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(GRU, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), self.hidden_dim).requires_grad_()
        out, (hn) = self.gru(x, (h0.detach()))
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
import os
BUDGET = 1000
API_KEY = os.environ.get("API_KEY", "")
API_SECRET = os.environ.get("API_SECRET", "")
BASE_URL = "https://paper-api.alpaca.markets"
ALPACA_CREDS = {
    "API_KEY": API_KEY,
    "API_SECRET": API_SECRET,
    "PAPER": True,
}
start_date = datetime(2020,1,1)
end_date = datetime(2023,12,31)




In [ ]:

class MLPTrader(Strategy):
    def initialize(self, cash_at_risk: float = 0.5):
        self.sleeptime = "24H"
        self.last_trade = None
        self.cash_at_risk = cash_at_risk
        self.symbol= "AAPL"
        self.api = REST(base_url=BASE_URL, key_id=API_KEY, secret_key=API_SECRET)
    def getStockPrices(self, stock):
        data = pd.read_feather(f'./AllData/{stock}.feather')
        data.index = pd.to_datetime(data.index).strftime('%Y-%B-%d')
        target_loc = (data.index.get_loc(self.get_datetime().strftime('%Y-%B-%d')))
        start_loc = max(0, target_loc - 19)
        selected_rows = data.iloc[start_loc:target_loc + 1]
        reshapedRows = (selected_rows['Close'].values.reshape(-1,1))
        return reshapedRows
    def forwardPass(self, data, stock):
        scaler = MinMaxScaler(feature_range=(-1,1))
        model = GRU(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers).to("cpu")
        model.load_state_dict(torch.load(f'./{stock}.pt'))
        model.eval()
        return scaler.inverse_transform(model(torch.from_numpy((np.array(([scaler.fit_transform(data)])))).type(torch.Tensor).to('cpu')).detach().numpy())
    def calculateBuyOrSell(self, results):
        growthStandings = results[0] - self.get_last_price(self.symbol)
        cash = self.get_cash()
        last_price = self.get_last_price(self.symbol)
        quantity = round(cash * self.cash_at_risk / last_price,0)
        if(growthStandings < 0):
            return False, quantity, last_price, cash
        else:
            return True, quantity, last_price, cash
    def on_trading_iteration(self):
        data = self.getStockPrices('AAPL')
        results = self.forwardPass(data, "AAPL")
        BuyOrSell, quantity, last_price, cash =self.calculateBuyOrSell(results)
        if cash > last_price:
            if(BuyOrSell):
                if self.last_trade == "sell": 
                    self.sell_all() 
                quantity = BuyOrSell[1]
                last_price = BuyOrSell[2]
                order = self.create_order(
                    self.symbol, 
                    quantity, 
                    "buy", 
                    type="bracket", 
                    take_profit_price=last_price*1.20, 
                    stop_loss_price=last_price*.95
                )
                self.submit_order(order)
                self.last_trade = 'buy'
            else:
                if self.last_trade == "buy": 
                    self.sell_all() 
                order = self.create_order(
                    self.symbol, 
                    quantity, 
                    "sell", 
                    type="bracket", 
                    take_profit_price=last_price*.8, 
                    stop_loss_price=last_price*1.05
                )
                self.submit_order(order) 
                self.last_trade = "sell"

In [ ]:
# Instantiate Trader\

trader = Trader()
broker = Alpaca(ALPACA_CREDS)
strategy = MLPTrader(name="MLPTrader", broker=broker, budget=BUDGET, symbol="SPY")

# Start BackTrading

backtesting_start = datetime(2023, 12, 1)
backtesting_end = datetime(2023, 12, 31)
strategy.backtest(YahooDataBacktesting, backtesting_start, backtesting_end, parameters={
    "cash_at_risk": 0.5,
})


In [ ]:
import torch
from torch import nn
from sklearn.preprocessing import MinMaxScaler
input_dim = 1
hidden_dim = 32
num_layers = 2
output_dim = 1
num_epochs = 100
def getStockPrices(stock, stockPrices):
        try:
            allValues = []
            scaler = MinMaxScaler(feature_range=(-1,1))
            reshapedRows = scaler.fit_transform(stockPrices[stock].values.reshape(-1,1))
            for index in range(len(reshapedRows)):
                 if(len(reshapedRows[index-20:index]) != 20):
                      continue
                 allValues.append(reshapedRows[index-20:index])
            array = np.array(allValues)
            data = torch.from_numpy(array).type(torch.Tensor)
            return (data, scaler)
        except Exception as e:
            print(e)
            return (None, None)
class GRU(nn.Module):
      def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
          super(GRU, self).__init__()
          self.hidden_dim = hidden_dim
          self.num_layers = num_layers

          self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True)
          self.fc = nn.Linear(hidden_dim, output_dim)

      def forward(self, x):
          h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).requires_grad_().to('cuda')
          out, (hn) = self.gru(x, (h0.detach()))
          out = self.fc(out[:, -1, :])
          return out
model = GRU(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers).to("cuda")
model.load_state_dict(torch.load(f'./FULL2.pt'))

In [ ]:
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
stockPriceValues = pd.read_feather("./FULLTESTER.feather")
data = stockPriceValues.loc[(stockPriceValues.index.tz_convert(None) > np.datetime64(pd.DatetimeIndex([datetime(2024,1,1)-timedelta(days=365)])[0])) & (stockPriceValues.index.tz_convert(None) < np.datetime64(pd.DatetimeIndex([datetime(2024,1,1)-timedelta(days=1)])[0]))]['Close']

In [ ]:
stock = 'ARVR'
results = getStockPrices(stock, data)

In [ ]:
results

In [ ]:
modelResults = model(results[0].to('cuda')).to('cpu').detach().numpy()

In [ ]:
results = results[1].inverse_transform(modelResults.reshape(-1,1))

In [ ]:
import matplotlib.pyplot as pl
values = data[stock].to_numpy()
values = values[~np.isnan(values)]
values=values[19:]
pl.plot(results)
pl.plot(np.array(values))

In [ ]:
results[-2]

In [ ]:
results = getStockPrices('GOOGL', data)
modelResults = model(results[0].to('cuda')).to('cpu').detach().numpy()
results = results[1].inverse_transform(modelResults.reshape(-1,1))
import matplotlib.pyplot as pl
values = data['GOOGL'].to_numpy()
values = values[~np.isnan(values)]
values=values[19:]
pl.plot(results)
pl.plot(np.array(values))


In [ ]:
results